# EmbedMed

## Setup

# ------ 

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

from controllers.ProcessController import ProcessController

config = {
    "GROQ_API_KEY": os.getenv("GROQ_API_KEY"),
    "GENERATION_MODEL": "openai/gpt-oss-120b",
    "EMBEDDING_MODEL": "BAAI/bge-small-en-v1.5",
    "VECTOR_DB_PATH": "chroma_db",
}

controller = ProcessController(config)


c:\Users\ALNOUR\anaconda3\envs\sic\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
controller.load_pdf(
    pdf_path="assets/hypertension-in-adults-diagnosis-and-management.pdf",
    document_id="NICE-NG136-2026",
    title="Hypertension in Adults: Diagnosis and Management",
    version="NG136",
    publication_date="2019-08-28"
)

controller.chunk_documents(document_id="NICE-NG136-2026")

controller.build_vectorstore(collection_name="hypertension_clinical_kb")

print("Pipeline completed successfully")
print(f"Pages loaded: {len(controller.pages)}")
print(f"Chunks created: {len(controller.chunks)}")

Pipeline completed successfully
Pages loaded: 52
Chunks created: 156


In [3]:
result = controller.ask("What is the target blood pressure for people with type 2 diabetes?")
print(result["answer"])
print("\nSources:")
for s in result["retrieved_sources"]:
    print(s)

The current NICE guidance sets the clinic blood‑pressure target for people with type 2 diabetes at **below 140 mm Hg systolic and below 90 mm Hg diastolic** (the same target used for hypertension in adults < 80 years)【NICE-NG136-2026 | p. 14 | NICE-NG136-2026-CH-038】.  

The committee noted that there is no robust evidence that lower targets (e.g., < 130/80 mm Hg) provide additional cardiovascular benefit in type 2 diabetes, and that using a slightly higher target may reduce adverse events and treatment burden【NICE-NG136-2026 | p. 38 | NICE-NG136-2026-CH-113】【NICE-NG136-2026 | p. 40 | NICE-NG136-2026-CH-119】.  

**Educational information only; not a diagnosis or medical advice.**

Sources:
{'document_id': 'NICE-NG136-2026', 'page': 38, 'chunk_id': 'NICE-NG136-2026-CH-113', 'preview': 'on people already receiving treatment and that it lacked information on adverse events.  The committee agreed that there was no evidence to suggest that blood pressure targets  sho'}
{'document_id': 'NICE

In [4]:
import sys
sys.path.append("evaluation")

from evaluation.eval_questions import EVAL_QUESTIONS
from evaluation.run_evaluation import run_retrieval

sample_results = run_retrieval(controller, EVAL_QUESTIONS[:2], k=3)

for r in sample_results:
    print("Question:", r["question"])
    for chunk in r["retrieved_chunks"]:
        print(f"  [{chunk['chunk_id']}] score={chunk['score']} page={chunk['page']}")
        print(f"  {chunk['chunk_text'][:100]}...")
    print()

Question: What is the target blood pressure for adults under 80 without diabetes?
  [NICE-NG136-2026-CH-038] score=0.8347 page=14
  hypertension in pregnancy. 
See also table 1 for clinic blood pressure targets for people aged under...
  [NICE-NG136-2026-CH-044] score=0.8283 page=16
  below 140/90 mmHg and ensure that it is maintained below that level. See also 
table 1 for guidance ...
  [NICE-NG136-2026-CH-115] score=0.8245 page=39
  people without hypertension. They also had concerns about the relevance of the study 
design. The co...

Question: What is the target blood pressure for adults aged 80 and over?
  [NICE-NG136-2026-CH-044] score=0.851 page=16
  below 140/90 mmHg and ensure that it is maintained below that level. See also 
table 1 for guidance ...
  [NICE-NG136-2026-CH-045] score=0.8431 page=16
  hypertension, use the average blood pressure level taken during the person's 
usual waking hours (se...
  [NICE-NG136-2026-CH-077] score=0.8146 page=28
  3 Blood pressure targets 

In [ ]:
sample_questions = EVAL_QUESTIONS[:3]
k_comparison = compare_k_values(controller, sample_questions)

for question, k_results in k_comparison.items():
    print("=" * 80)
    print("Question:", question)
    for k, chunks in k_results.items():
        print(f"\n  --- k={k} ({len(chunks)} results) ---")
        for c in chunks:
            print(f"  [{c['chunk_id']}] page={c['page']} score={c['score']}")